# XDMoD Data Analytics Framework — XDMoD Analysis

v1.0.1

© 2024 University at Buffalo Center for Computational Research

See the [xdmod-notebooks](https://github.com/ubccr/xdmod-notebooks) repository for information on setup, support, contributing, licensing, and referencing.

## Introduction

The XDMoD Data Analytics Framework provides API access to the data in XDMoD via the [`xdmod_data` Python module](https://pypi.org/project/xdmod-data). This notebook provides real-world examples the capabilities of XDMoD Data. You will use interactive elements (Jupyter Widgets) to customize and retrieve data that comes from XDMoD.  

## Install/upgrade the required modules

Run the code below to install/upgrade the modules needed to run this notebook.

In [1]:
import sys
! {sys.executable} -m pip install --upgrade 'xdmod-data>=1.0.0,<2.0.0' python-dotenv tabulate

If running that code caused a new version of Plotly to be installed/upgraded, you may need to refresh your browser window for plots to appear correctly.

## Configure notebook formatting

### Exceptions

Run the code below to simplify how Python exceptions are displayed in this notebook.

In [2]:
import sys
def exception_handler(exception_type, exception, traceback):
    print("%s: %s" % (exception_type.__name__, exception), file=sys.stderr)
get_ipython()._showtraceback = exception_handler

### Tables

Run the code below to set up for displaying Pandas DataFrames as Markdown tables in this notebook.

In [3]:
from IPython.display import display, Markdown
def display_df_md_table(df):
    return display(Markdown(df.replace('\n', '<br/>', regex=True).to_markdown()))

### Plots

Run the code below to set up the external Plotly library to make plots using a custom XDMoD theme.

In [4]:
import plotly.express as px
import plotly.io as pio
import xdmod_data.themes
pio.templates.default = "timeseries"

## Create an environment file

The `xdmod-data.env` file will store your XDMoD API token.

Run the code below to create the file in your home directory (if it does not already exist) and allow only you to read and write to it.

In [5]:
from pathlib import Path
from os.path import expanduser
xdmod_data_env_path = Path(expanduser('~/xdmod-data.env'))
try:
    with open(xdmod_data_env_path):
        pass
except FileNotFoundError:
    with open(xdmod_data_env_path, 'w') as xdmod_data_env_file:
        xdmod_data_env_file.write('XDMOD_API_TOKEN=')
    xdmod_data_env_path.chmod(0o600)

## Obtain an API token

Follow [these instructions](https://github.com/ubccr/xdmod-data#api-token-access) to obtain an API token.

## Store your API token in the environment file

Open the `xdmod-data.env` file and paste your token after `XDMOD_API_TOKEN=`.

Save the file.

## Load your XDMoD API token into the environment

Run the code below to load the contents of the `xdmod-data.env` file into the environment. It will print `True` if it successfully loaded the file.

In [6]:
from dotenv import load_dotenv
load_dotenv(xdmod_data_env_path, override=True)

True

## Initialize the XDMoD Data Warehouse

Run the code below to prepare for getting data from the XDMoD data warehouse at the given URL.

In [7]:
from xdmod_data.warehouse import DataWarehouse
dw = DataWarehouse('https://xdmod.access-ci.org')

## Prepare for interactive elements

Run the code below to set up the external `Jupyter’s widgets` library to use interactive elements for analyzing data.

In [8]:
pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [9]:
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from IPython.display import clear_output

## Initial customization settings 

customize the settings that you want throughout the entire rest of the notebook

In [10]:
start= widgets.DatePicker(
    description='Pick a Start Date'
)

end = widgets.DatePicker(
    description='Pick a End Date',
)

display(start)
display(end)

DatePicker(value=None, description='Pick a Start Date', step=1)

DatePicker(value=None, description='Pick a End Date', step=1)

In [16]:
with dw:
    data1 = dw.get_filter_values('Jobs','Service Provider')
    dur_data = dw.get_durations()
    agg_data = dw.get_aggregation_units()

    
list1 = data1['label'].tolist()
dict1= {}
for label in list1:
    dict1[label]= False
    
dur_list = []
for i in dur_data:
    dur_list.append(i)


agg_list = []
for n in agg_data:
    agg_list.append(n)



def func(**wag):
    # pass
    # print(locals())
    global diction 
    diction = locals()
    diction = (diction['wag'])
    print(diction)
    plist=[]
    for providers, values in diction.items():
        if values:
            plist.append(providers)
    func2(plist)

def func2(providers_list):
    with dw:
        rdata = dw.get_data(
            duration=(start.value, end.value),
            realm='Jobs',
            metric='CPU Hours: Total',
            filters={'provider':providers_list},
            dimension='resource'
            
        )
    resource_list = rdata.columns.tolist()
    global rdict1
    rdict1 = {}
    for label in resource_list:
        rdict1[label]= False
    func_display1(rdict1)

def func4(**wag):
    unfiltered_resources = locals()
    unfiltered_resources = unfiltered_resources['wag']
    global filtered_resource_list
    filtered_resource_list = []
    for picked_resources, values in unfiltered_resources.items():
        if values:
            filtered_resource_list.append(picked_resources)
    plot_charts(filtered_resource_list)
            

    


def func_display1(rdict1):
    for i in range(0,len(accordion.children)):
        if i == 2:
            accordion.children[2].close()
    y = interactive(func4,**rdict1)
    accordion.children = [*accordion.children,widgets.VBox([y])]
    accordion.set_title(2,'Resources')
    print(rdict1)
    


dropdown = widgets.Dropdown(
    options=dur_list,
    value='Previous month',
    description='Duration:',
    disabled=False,
    width = 100
)

radio_button = widgets.RadioButtons(
    options=agg_list,
    value='Auto',
    description='Aggregation Units: ',
    disabled=False)

def printss(change):
    print(change)


global rdict1
rdict1 = {}
x = interactive(func,**dict1)
print(rdict1)
#y = interactive(func4,**rdict1)
func_dropdown= interactive(printss,change=dropdown)
func_radio_buttons= interactive(printss,change=radio_button)
z = [func_dropdown,func_radio_buttons]


lists_widgets=[widgets.HBox([z[0],z[1]]),widgets.VBox([x])]
global accordion
accordion = widgets.Accordion(children=lists_widgets)
accordion.set_title(0,'Duration')
accordion.set_title(1, 'Resource Providers')
display(accordion)
print(filtered_resource_list)



{'Purdue Anvil CPU': False, 'Expanse': False, 'Bridges 2 RM': False, 'TACC STAMPEDE3': False, 'NCSA DELTA GPU': False, 'NCSA DELTA CPU': False, 'STAMPEDE2 TACC': False, 'Texas A&M U FASTER': False, 'JHU Rockfish RM': False, 'Bridges 2 GPU': False, 'Expanse GPU': False, 'Purdue Anvil GPU': False, 'UD DARWIN': False, 'IACS Ookami': False, 'Bridges 2 EM': False, 'Bridges2 GPU AI': False, 'UD DARWIN GPU': False, 'KyRIC': False, 'PSC Neocortex': False, 'neocortex sdflex.psc.access ci.org': False}


Accordion(children=(HBox(children=(interactive(children=(Dropdown(description='Duration:', index=5, options=('…

[]


In [15]:
def plot_charts(rlist2):
    with dw:
        data = dw.get_data(
            duration=(start.value, end.value),
            realm='Jobs',
            metric='CPU Hours: Total',
            dimension = 'Resource',
            filters={'resource':rlist2},
            #aggregation_unit= agg_unit
        )
        plot = px.line(data, labels={'value': 'CPU Hours: Total'})
        plot.show()

#plot_charts(filtered_resource_list)


